# 03 특징 엔지니어링

주간 type×family 패널에 lag/rolling 및 달력·외생 변수 피처를 생성합니다.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))
from utils.paths import DATA_PROCESSED

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
dfw = dfw.sort_values(['type','family','yearweek']).reset_index(drop=True)
dfw.head()



,type,family,year,week,yearweek,sales,onpromotion,transactions,dcoilwtico,is_holiday
0,A,AUTOMOTIVE,2013,1,201301,430.0,0,139419.0,93.110000,True
1,A,AUTOMOTIVE,2013,2,201302,582.0,0,162919.0,93.538571,False
2,A,AUTOMOTIVE,2013,3,201303,582.0,0,161440.0,94.927143,False
3,A,AUTOMOTIVE,2013,4,201304,621.0,0,157677.0,95.531429,False
4,A,AUTOMOTIVE,2013,5,201305,663.0,0,165483.0,97.190000,False


In [2]:
# lag / rolling (주간)
for lag in [1, 2, 4, 8]:
    dfw[f'lag_{lag}'] = dfw.groupby(['type','family'])['sales'].shift(lag)
for w in [4, 8, 12]:
    dfw[f'roll_mean_{w}'] = dfw.groupby(['type','family'])['sales'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    dfw[f'roll_std_{w}'] = dfw.groupby(['type','family'])['sales'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).std())

dfw['zero_ratio_12'] = dfw.groupby(['type','family'])['sales'].transform(
    lambda s: s.shift(1).rolling(12, min_periods=1).apply(lambda x: (x == 0).mean())
)
dfw.head()



,type,family,year,week,yearweek,sales,onpromotion,transactions,dcoilwtico,is_holiday,...,lag_2,lag_4,lag_8,roll_mean_4,roll_std_4,roll_mean_8,roll_std_8,roll_mean_12,roll_std_12,zero_ratio_12
0,A,AUTOMOTIVE,2013,1,201301,430.0,0,139419.0,93.110000,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A,AUTOMOTIVE,2013,2,201302,582.0,0,162919.0,93.538571,False,...,NaN,NaN,NaN,430.000000,NaN,430.000000,NaN,430.000000,NaN,0.0
2,A,AUTOMOTIVE,2013,3,201303,582.0,0,161440.0,94.927143,False,...,430.0,NaN,NaN,506.000000,107.480231,506.000000,107.480231,506.000000,107.480231,0.0
3,A,AUTOMOTIVE,2013,4,201304,621.0,0,157677.0,95.531429,False,...,582.0,NaN,NaN,531.333333,87.757241,531.333333,87.757241,531.333333,87.757241,0.0
4,A,AUTOMOTIVE,2013,5,201305,663.0,0,165483.0,97.190000,False,...,582.0,430.0,NaN,553.750000,84.523665,553.750000,84.523665,553.750000,84.523665,0.0


In [3]:
# 달력 피처
dfw['month'] = ((dfw['yearweek'] % 100) // 4 + 1).clip(1, 12)  # rough month proxy from week
# yearweek에서 연도
dfw['year_feat'] = dfw['yearweek'] // 100

feat_cols = [c for c in dfw.columns if c not in ['sales']]
print('feature columns:', len(feat_cols))
dfw[feat_cols].head()



feature columns: 22


,type,family,year,week,yearweek,onpromotion,transactions,dcoilwtico,is_holiday,lag_1,...,lag_8,roll_mean_4,roll_std_4,roll_mean_8,roll_std_8,roll_mean_12,roll_std_12,zero_ratio_12,month,year_feat
0,A,AUTOMOTIVE,2013,1,201301,0,139419.0,93.110000,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2013
1,A,AUTOMOTIVE,2013,2,201302,0,162919.0,93.538571,False,430.0,...,NaN,430.000000,NaN,430.000000,NaN,430.000000,NaN,0.0,1,2013
2,A,AUTOMOTIVE,2013,3,201303,0,161440.0,94.927143,False,582.0,...,NaN,506.000000,107.480231,506.000000,107.480231,506.000000,107.480231,0.0,1,2013
3,A,AUTOMOTIVE,2013,4,201304,0,157677.0,95.531429,False,582.0,...,NaN,531.333333,87.757241,531.333333,87.757241,531.333333,87.757241,0.0,2,2013
4,A,AUTOMOTIVE,2013,5,201305,0,165483.0,97.190000,False,621.0,...,NaN,553.750000,84.523665,553.750000,84.523665,553.750000,84.523665,0.0,2,2013


In [4]:
out = DATA_PROCESSED / 'df_weekly_features.parquet'
dfw.to_parquet(out, index=False)
print('저장:', out, '| shape:', dfw.shape)



저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\df_weekly_features.parquet | shape: (39930, 23)


## 분석 요약

### 생성 피처 (총 22개, sales 제외)
| 구분 | 피처 |
|------|------|
| **Lag** | `lag_1`, `lag_2`, `lag_4`, `lag_8` (1~8주 전 판매) |
| **Rolling** | `roll_mean/std_{4,8,12}` (과거 이동평균·표준편차) |
| **간헐성** | `zero_ratio_12` (최근 12주 0판매 비율) |
| **외생·달력** | `onpromotion`, `transactions`, `dcoilwtico`, `is_holiday`, `month`, `year_feat` |

### 데이터 형태
- **39,930행 × 23컬럼** (165 시계열 × 242주)
- lag 피처는 시계열 첫 주에 결측 165건(시리즈당 1건) — 정상적인 shift 결과

### 활용
- `07` ML 예측(RF, XGBoost)의 패널 학습 피처로 사용
- rolling·zero_ratio는 **간헐 수요(Lumpy/Intermittent)** 패턴 포착에 유리